# Agentic RAG — قانون الالتزامات والعقود المغربي
**Dahir 9 Ramadan 1331 — Ministere de la Justice — 268 pages**

## Flux
```
CELL 1   Installation
CELL 2   Upload PDF + OCR Tesseract parallele  ->  ocr_pages (dict)
CELL 3   Nettoyage OCR + Parsing articles      ->  articles (list[LegalArticle])
CELL 4   Arabic BERT + FAISS GPU               ->  embed_model, index
CELL 5   BM25 + Hybrid Retriever (RRF)         ->  retriever
CELL 6   Qwen2.5-7B-Instruct GGUF              ->  llm
CELL 7   Agentic RAG Engine                    ->  rag
CELL 8   Tests use cases juridiques
CELL 9   Interface interactive
CELL 10  Evaluation Precision@K & Recall@K
```
> **Pourquoi OCR ?** Polices CID ToUnicode cassees -> PyMuPDF produit du garbage.
> Tesseract lang=ara sur JPEG 300 DPI = arabe propre.

**Runtime : GPU T4 | VRAM : ~6 GB Qwen + ~2 GB BERT**

## CELL 1 - Installation des dependances

In [2]:
# CELL 1 - Installation (executer une seule fois par session)
import subprocess

def run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'OK' if r.returncode==0 else 'ERR'} | {label or cmd[:65]}")
    if r.returncode != 0: print(f"   {r.stderr[-200:]}")

run("apt-get install -y -q tesseract-ocr tesseract-ocr-ara poppler-utils",
    "tesseract-ocr Arabic + poppler-utils")
run("pip install -q pytesseract Pillow", "pytesseract + Pillow")
run("pip install -q transformers sentence-transformers", "transformers")
run("pip install -q torch --index-url https://download.pytorch.org/whl/cu118", "torch CUDA")
run("pip install -q faiss-gpu-cu11", "faiss-gpu")
run("pip install -q rank_bm25", "rank_bm25")
run("pip install -q llama-cpp-python "
    "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122",
    "llama-cpp-python CUDA")
run("pip install -q huggingface_hub tqdm", "huggingface_hub + tqdm")
print("\nInstallation complete — passez a CELL 2")

OK | tesseract-ocr Arabic + poppler-utils
OK | pytesseract + Pillow
OK | transformers
OK | torch CUDA
OK | faiss-gpu
OK | rank_bm25
OK | llama-cpp-python CUDA
OK | huggingface_hub + tqdm

Installation complete — passez a CELL 2


## CELL 2 - Upload PDF + OCR Tesseract parallele

In [ ]:
# CELL 2 - Upload PDF + OCR parallele avec cache
#
# Produit  : ocr_pages  dict {int_page_num: str_texte_arabe}
# Cache    : /content/loc_ocr_pages.json  (recharge auto si present)
# Temps    : ~25-40 min Colab 2 CPU, 268 pages

import os, json, glob, time, subprocess, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from PIL import Image
import pytesseract
from google.colab import files
from tqdm.notebook import tqdm

# Chemins globaux (utilises dans toutes les cellules suivantes)
OCR_CACHE      = "/content/drive/MyDrive/3D_SMART/loc_ocr_pages.json"
ARTICLES_CACHE = "/content/drive/MyDrive/3D_SMART/loc_articles.json"
EMBED_CACHE    = "/content/drive/MyDrive/3D_SMART/loc_embeddings.npy"
IMG_DIR        = "/content/drive/MyDrive/3D_SMART/loc_pages"
os.makedirs(IMG_DIR, exist_ok=True)

# Upload
PDF_PATH = "/content/drive/MyDrive/3D_SMART/lois/les_lois_des_obligations_et_contrats.pdf"
if not os.path.exists(PDF_PATH):
    print("Upload le PDF du LOC :")
    uploaded = files.upload()
    PDF_PATH = f"/content/{list(uploaded.keys())[0]}"
print(f"PDF : {PDF_PATH}")

def rasterize_pdf(pdf_path, out_dir, dpi=300):
    existing = sorted(glob.glob(f"{out_dir}/page-*.jpg"))
    if existing:
        print(f"Images deja presentes : {len(existing)} pages")
        return existing
    print(f"Rasterisation a {dpi} DPI...")
    r = subprocess.run(
        ["pdftoppm", "-jpeg", "-r", str(dpi), pdf_path, f"{out_dir}/page"],
        capture_output=True, text=True
    )
    if r.returncode != 0: raise RuntimeError(r.stderr)
    pages = sorted(glob.glob(f"{out_dir}/page-*.jpg"))
    print(f"{len(pages)} pages rasterisees")
    return pages

def _ocr_one_page(args):
    path, num = args
    text = pytesseract.image_to_string(
        Image.open(path), lang="ara",
        config="--psm 6 --oem 3 -c preserve_interword_spaces=1"
    )
    return num, text.strip()

def run_ocr_parallel(page_files, n_workers=None):
    if os.path.exists(OCR_CACHE):
        print(f"Cache OCR : {OCR_CACHE}")
        with open(OCR_CACHE, "r", encoding="utf-8") as f:
            return {int(k): v for k, v in json.load(f).items()}
    n_workers = n_workers or multiprocessing.cpu_count()
    args = [(f, i+1) for i, f in enumerate(page_files)]
    results = {}
    est = len(page_files) * 17 / n_workers / 60
    print(f"OCR : {len(page_files)} pages | {n_workers} workers | ~{est:.0f} min")
    start = time.time()
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(_ocr_one_page, a): a[1] for a in args}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="OCR pages"):
            pn, tx = fut.result()
            results[pn] = tx
    print(f"OCR termine en {(time.time()-start)/60:.1f} min")
    with open(OCR_CACHE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Cache sauvegarde : {OCR_CACHE}")
    return results

page_files = rasterize_pdf(PDF_PATH, IMG_DIR, dpi=300)
ocr_pages  = run_ocr_parallel(page_files)
print(f"\nocr_pages : {len(ocr_pages)} pages chargees")
print("\n--- Apercu page 5 ---")
print(ocr_pages.get(5, "")[:400])

## CELL 3 - Nettoyage OCR + Parsing des articles

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# =====================================================================
# CELL 3 — Nettoyage OCR + Parsing des articles
#
# Ce que cette cellule fait:
#   1. Définit toutes les fonctions de nettoyage (clean_page, etc.)
#   2. Définit LegalArticle et LOCParser
#   3. Charge le JSON OCR, nettoie, parse -> variable `articles`
#
# Variable produite pour les cellules suivantes:
#   articles : list[LegalArticle]  (1258 articles du LOC)
# =====================================================================

import re, os
import json
from dataclasses import dataclass, asdict
from typing import Optional

# ─────────────────────────────────────────────────────────────────
# PARTIE A — Filtres de nettoyage par ligne
# ─────────────────────────────────────────────────────────────────

# Déclencheur HARD de note de bas de page.
# ATTENTION : "1 - الأهلية للالتزام" = CONTENU d'article (liste numérotée) -> NE PAS filtrer
# On filtre UNIQUEMENT les lignes contenant des verbes de référence légale
# qui n'apparaissent JAMAIS dans le corps d'un article.
_HARD_FOOTNOTE = re.compile(
    r"""^\d{1,3}\s*[-\u2013]\s*
    (?:
        ورد\s+في\s+النص       |
        وردت\s+في\s+النص      |
        قارن\s+مع\s+          |
        انظر\s+(?:الفصل|المادة|ظهير)    |
        تمم\s+(?:القسم|الفصل|الباب)     |
        تممت\s+               |
        أضاف\s+               |
        أضيف\s+               |
        تم\s+تغيير            |
        تم\s+تتميم            |
        تم\s+إضافة            |
        المادة\s+\d+\s+من\s+(?:مدونة|القانون|قانون)  |
        مقارنة\s+مع\s+النص    |
        سقطت\s+من\s+الترجمة   |
        تتحدث\s+بعض\s+فصول   |
        المقصود\s+بالقانون
    )""",
    re.VERBOSE | re.UNICODE,
)

# Lignes de continuation de notes (références Bulletin Officiel, dahirs)
_FOOTNOTE_CONTINUATION = re.compile(
    r"""(?:
        الجريدة\s+الرسمية              |
        ظهير\s+شريف\s+رقم             |
        بتنفيذه\s+ظهير                |
        صادر\s+في\s+\d+\s+من          |
        بتاريخ\s+\d+\s+من\s+(?:ذي|ربيع|شعبان|رمضان|شوال|ذو|جمادى|محرم|صفر)  |
        \bص\s*\d{3,}\b                |
        عدد\s+\d{4,}\s+بتاريخ         |
        رقم\s+[\d.]+\s+(?:يتعلق|بمثابة|المتعلق)
    )""",
    re.VERBOSE | re.UNICODE,
)

_ARTICLE_HEADER = re.compile(r"^الفصل\s*[\d]")


def _is_page_header(s):
    """En-tête ministériel haut de page (39 variantes OCR couvertes)."""
    return bool(
        re.search(r"(?:المملكة|الدسايكة|الساكة|المساكة|الم\s+اكد)", s)
        and re.search(r"(?:وزارة|العذز|العذل|العذ\b|مكيرية|مديرية)", s)
    )

def _is_page_number(s):
    return bool(re.match(r"^[-\u2013\s]*\d{1,3}[-\u2013\s]*$", s))

def _is_stray_number(s):
    return bool(re.match(r"^\d{1,3}$", s))

def _is_garbled_latin(s):
    """Ratio caractères latins > 45% = citation française OCR-ée."""
    if len(s) < 8:
        return False
    return len(re.findall(r"[a-zA-Z]", s)) / len(s) > 0.45

def _is_editorial_rewrite(s):
    """Reformulation éditoriale (correction de traduction, pas du texte de loi)."""
    return bool(re.search(r"وبذلك\s+يمكن\s+(?:صياغة|تحديد)", s))


def clean_page(text):
    """
    Nettoie une page OCR du LOC.

    Supprime : en-tête ministériel, numéros de page, notes de bas de page,
    lignes de références JO, garbled Latin, reformulations éditoriales.

    Conserve : corps des articles, listes numérotées DANS les articles
    (ex: "1 - الأهلية للالتزام"), titres structurels.
    """
    lines = text.split("\n")
    clean = []
    in_footnote_zone = False

    for line in lines:
        s = line.strip()
        if not s:
            continue

        # Filtres absolus
        if _is_page_header(s):      continue
        if _is_page_number(s):      continue
        if _is_stray_number(s):     continue
        if _is_garbled_latin(s):    continue
        if _is_editorial_rewrite(s):
            in_footnote_zone = True
            continue

        # Nouveau article → reset zone notes
        if _ARTICLE_HEADER.match(s):
            in_footnote_zone = False

        # Déclencheur HARD de note (verbes de référence légale uniquement)
        if _HARD_FOOTNOTE.match(s):
            in_footnote_zone = True

        if in_footnote_zone:
            continue

        # Référence légale hors-zone (sécurité supplémentaire)
        if re.match(r"^\d{1,3}\s*[-\u2013]", s) and _FOOTNOTE_CONTINUATION.search(s):
            continue

        clean.append(s)

    return "\n".join(clean)


# ─────────────────────────────────────────────────────────────────
# PARTIE B — Normalisation du texte
# ─────────────────────────────────────────────────────────────────

_OCR_QUOTES  = re.compile(r'[""\u201c\u201d]')
_SUPERSCRIPT = re.compile(r'([\u0600-\u06FF])[\'\d]{1,3}(?=\s|$|[،؛.])')
_MULTI_SPACE = re.compile(r'[ \t]{2,}')


def normalize_line(s):
    """Normalisation légère : guillemets OCR, superscripts parasites, espaces."""
    s = _OCR_QUOTES.sub("", s)
    s = _SUPERSCRIPT.sub(r"\1", s)
    s = _MULTI_SPACE.sub(" ", s)
    return s.strip()


# ─────────────────────────────────────────────────────────────────
# PARTIE C — Correction des numéros d'articles contaminés
#
# OCR colle parfois le chiffre superscript au numéro d'article :
#   "الفصل2901197" = "الفصل 290" + référence "1197"
#   "الفصل 2.1197" = "الفصل 2.1" + référence "197"
#   "الفصل4277"   = "الفصل 427" + référence "7"
#
# Le LOC va de 1 à ~1250 (articles purs)
# Composites réels : X-Y (ex: 3-106, 618-20, 417-1) et X.Y (ex: 2.1)
# ─────────────────────────────────────────────────────────────────

_LOC_MAX = 1260


def clean_article_num(raw):
    """Récupère le vrai numéro d'article depuis un token OCR potentiellement contaminé."""
    s = raw.strip()

    # Cas X.Y (point — seulement "2.1" dans le LOC)
    m = re.match(r"^(\d+)\.(\d+)$", s)
    if m:
        x_str, y_str = m.group(1), m.group(2)
        x, y = int(x_str), int(y_str)
        if x <= _LOC_MAX:
            if y <= 9:
                return s                           # "2.1" → valide
            else:
                return f"{x_str}.{y_str[0]}"      # "2.1197" → "2.1"

    # Cas X-Y (tiret — composites réels du LOC)
    m = re.match(r"^(\d+)-(\d+)$", s)
    if m:
        x_str, y_str = m.group(1), m.group(2)
        x, y = int(x_str), int(y_str)
        if x <= _LOC_MAX and y <= _LOC_MAX:
            return s                               # "3-106", "617-20" → valide
        if x <= _LOC_MAX:
            for l in range(len(y_str) - 1, 0, -1):
                if int(y_str[:l]) <= _LOC_MAX:
                    return f"{x_str}-{y_str[:l]}"
        for l in range(len(x_str) - 1, 0, -1):
            if 0 < int(x_str[:l]) <= _LOC_MAX:
                return f"{x_str[:l]}-{y_str[:3] if len(y_str) > 3 else y_str}"

    # Cas entier pur
    m = re.match(r"^(\d+)$", s)
    if m:
        n_str = m.group(1)
        if int(n_str) <= _LOC_MAX:
            return s
        for l in range(len(n_str) - 1, 0, -1):
            if 0 < int(n_str[:l]) <= _LOC_MAX:
                return n_str[:l]

    return s  # fallback


# ─────────────────────────────────────────────────────────────────
# PARTIE D — Dataclass LegalArticle
# ─────────────────────────────────────────────────────────────────

@dataclass
class LegalArticle:
    """
    Un article du LOC après nettoyage.

    Champs :
        article_num     → numéro textuel ("1", "2.1", "417-1", "618-20")
        article_num_int → premier entier pour tri/filtre
        text            → corps de l'article nettoyé
        book            → الكتاب courant
        section         → القسم / الباب courant
        subsection      → الفرع courant
        char_count      → longueur (pour déduplication)
    """
    article_num:     str
    article_num_int: int
    text:            str
    book:            str = ""
    section:         str = ""
    subsection:      str = ""
    char_count:      int = 0

    def to_chunk_text(self):
        """
        Texte enrichi pour l'embedding.
        Format : [الفصل N] [الكتاب X] [القسم Y]\n<corps>
        """
        header = f"[الفصل {self.article_num}]"
        if self.book:       header += f" [{self.book}]"
        if self.section:    header += f" [{self.section}]"
        if self.subsection: header += f" [{self.subsection}]"
        return f"{header}\n{self.text}"


# ─────────────────────────────────────────────────────────────────
# PARTIE E — LOCParser
# ─────────────────────────────────────────────────────────────────

class LOCParser:
    """
    Parseur structuré du قانون الالتزامات والعقود.
    Pipeline : nettoyer → splitter sur الفصل N → extraire contexte → dédupliquer → trier.
    """

    _ART_SPLIT = re.compile(
        r"(?:^|\n)(الفصل\s*[\d]+(?:[.\-][\d]+)*)",
        re.MULTILINE,
    )
    _ART_NUM   = re.compile(r"الفصل\s*([\d]+(?:[.\-][\d]+)*)")
    _BOOK_RE   = re.compile(r"الكتاب\s+\S+(?:\s+\S+)?")
    _SECT_RE   = re.compile(r"(?:القسم|الباب)\s+\S+(?:\s+\S+)?")
    _SUB_RE    = re.compile(r"الفرع\s+\S+(?:\s+\S+)?")
    _HDR_NOISE = re.compile(
        r"(?:المملكة|الدسايكة).*?(?:التشريع|التشيوع|العشيوم)\n?",
        re.DOTALL,
    )

    @staticmethod
    def _to_int(s):
        m = re.search(r"\d+", s)
        return int(m.group()) if m else 0

    def parse(self, ocr_pages):
        """Parse depuis {page_num: raw_text}. Retourne list[LegalArticle]."""
        pages_sorted = sorted(ocr_pages.keys(), key=lambda k: int(k))
        full = "\n".join(clean_page(ocr_pages[k]) for k in pages_sorted)

        parts = self._ART_SPLIT.split(full)
        arts = []
        cur_book = cur_sect = cur_sub = ""

        if parts:
            bm = self._BOOK_RE.search(parts[0])
            if bm: cur_book = bm.group().strip()
            sm = self._SECT_RE.search(parts[0])
            if sm: cur_sect = sm.group().strip()

        i = 1
        while i < len(parts) - 1:
            header = parts[i].strip()
            body   = parts[i + 1] if i + 1 < len(parts) else ""

            bm = self._BOOK_RE.search(body)
            if bm: cur_book = bm.group().strip()
            sm = self._SECT_RE.search(body)
            if sm: cur_sect = sm.group().strip()
            sub = self._SUB_RE.search(body)
            if sub: cur_sub = sub.group().strip()

            nm = self._ART_NUM.search(header)
            if nm:
                num_str    = clean_article_num(nm.group(1))
                clean_body = self._HDR_NOISE.sub("", body).strip()
                clean_body = re.sub(r"\n{3,}", "\n\n", clean_body)
                clean_body = "\n".join(
                    normalize_line(ln)
                    for ln in clean_body.split("\n")
                    if ln.strip()
                )
                if len(clean_body) > 20:
                    arts.append(LegalArticle(
                        article_num=num_str,
                        article_num_int=self._to_int(num_str),
                        text=clean_body,
                        book=cur_book,
                        section=cur_sect,
                        subsection=cur_sub,
                        char_count=len(clean_body),
                    ))
            i += 2

        seen = {}
        for a in arts:
            if a.article_num not in seen or a.char_count > seen[a.article_num].char_count:
                seen[a.article_num] = a

        return sorted(seen.values(), key=lambda a: (a.article_num_int, a.article_num))

    def save(self, articles, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump([asdict(a) for a in articles], f, ensure_ascii=False, indent=2)
        print(f"Sauvegarde : {len(articles)} articles → {path}")

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            return [LegalArticle(**d) for d in json.load(f)]


# ─────────────────────────────────────────────────────────────────
# PARTIE F — Exécution : produire la variable `articles`
#
# C'est ici que `articles` est créé et rendu disponible
# pour toutes les cellules suivantes (4, 5, 6, 7...).
# ─────────────────────────────────────────────────────────────────

parser = LOCParser()

if os.path.exists(ARTICLES_CACHE):
    # Rechargement depuis cache (évite de re-parser à chaque session)
    articles = parser.load(ARTICLES_CACHE)
    print(f"Cache chargé : {len(articles)} articles depuis {ARTICLES_CACHE}")
else:
    # Premier lancement : parser depuis le JSON OCR
    print("Parsing depuis le JSON OCR...")
    with open(OCR_CACHE, "r", encoding="utf-8") as f:
        ocr_pages = json.load(f)
    articles = parser.parse(ocr_pages)
    parser.save(articles, ARTICLES_CACHE)

# ─────────────────────────────────────────────────────────────────
# Statistiques de vérification
# ─────────────────────────────────────────────────────────────────
total_chars = sum(a.char_count for a in articles)
print(f"\nRésultats :")
print(f"  Articles extraits  : {len(articles)}")
print(f"  Plage              : الفصل {articles[0].article_num} → الفصل {articles[-1].article_num}")
print(f"  Longueur moyenne   : {total_chars // len(articles)} chars/article")
print(f"  Livres détectés    : {len(set(a.book for a in articles if a.book))}")
short = sum(1 for a in articles if a.char_count < 50)
if short:
    print(f"  Articles courts    : {short} (vérifier manuellement)")

# Aperçu de 3 articles pour vérification visuelle
print("\n--- الفصل 2 ---")
a2 = next((a for a in articles if a.article_num == "2"), None)
if a2: print(a2.to_chunk_text())

print("\n--- الفصل 77 ---")
a77 = next((a for a in articles if a.article_num == "77"), None)
if a77: print(a77.to_chunk_text()[:300])

print("\n✅ Variable `articles` prête pour les cellules 4, 5, 6, 7")


Cache chargé : 1258 articles depuis /content/drive/MyDrive/3D_SMART/loc_articles.json

Résultats :
  Articles extraits  : 1258
  Plage              : الفصل 1 → الفصل 1250
  Longueur moyenne   : 262 chars/article
  Livres détectés    : 5
  Articles courts    : 15 (vérifier manuellement)

--- الفصل 2 ---
[الفصل 2] [الكتاب الأول منه] [الباب الأول: الالتزامات]
الأركان اللازمة لصحة الالتزامات الناشئة عن التعبير عن الإرادةهى:
1 - الأهليةللالتزام؛
2 - تعبير صحيح عن الإرادة يقع على العناصر الأساسيةللالتزام؛
4 - سبب مشروعللالتزام.

--- الفصل 77 ---
[الفصل 77] [الكتاب الأول من] [الباب الثالث: الالتزامات] [الفرع الرابع: أحكاممتفرقة]
كل فعل ارتكبه الإنسان عن بينة واختيارء ومن غير أن يسمح له به القانون»فأحدث
السبب المباشر في حصولالضرر.
وكل شرط مخالف لذلك يكون عديمالأثر.

✅ Variable `articles` prête pour les cellules 4, 5, 6, 7


## CELL 4 - Arabic BERT Embeddings + FAISS GPU

In [8]:
# SI vous avez déja le fichie , avec le lancement de ce code

# CELL 4 - Embeddings Arabic BERT + Index FAISS GPU
#
# Requiert : articles (Cell 3), EMBED_CACHE, os
# Produit  : embed_model, embeddings, index, DIM, np
#
# Modele : CAMeL-Lab/camel-bert-base-sts
#   Fine-tune Arabic Semantic Textual Similarity
#   768 dims, ~110M params, rapide sur T4
#   Meilleur que mBERT/LaBSE pour arabe mono-langue

import torch
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

print("\nChargement CAMeL-Lab/camel-bert-base-sts...")
# embed_model = SentenceTransformer("CAMeL-Lab/camel-bert-base-sts", device=DEVICE)
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device=DEVICE)


chunk_texts = [a.to_chunk_text() for a in articles]

if os.path.exists(EMBED_CACHE):
    embeddings = np.load(EMBED_CACHE)
    print(f"Embeddings depuis cache : {embeddings.shape}")
else:
    print(f"Encodage de {len(chunk_texts)} articles...")
    embeddings = embed_model.encode(
        chunk_texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    )
    np.save(EMBED_CACHE, embeddings)
    print(f"Embeddings sauvegardes : {embeddings.shape}")

DIM       = embeddings.shape[1]
cpu_index = faiss.IndexFlatIP(DIM)
cpu_index.add(embeddings.astype(np.float32))

if DEVICE == "cuda":
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    print(f"\nFAISS GPU : {index.ntotal} vecteurs x {DIM} dims")
else:
    index = cpu_index
    print(f"\nFAISS CPU : {index.ntotal} vecteurs x {DIM} dims")

print("embed_model et index prets pour Cell 5")

Device : cuda
  GPU  : Tesla T4
  VRAM : 15.6 GB

Chargement CAMeL-Lab/camel-bert-base-sts...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings depuis cache : (1258, 768)

FAISS GPU : 1258 vecteurs x 768 dims
embed_model et index prets pour Cell 5


## CELL 5 - BM25 + Hybrid Retriever (RRF Fusion)

In [9]:
# CELL 5 - BM25 sparse + Hybrid Retriever RRF
#
# Requiert : articles, index, embed_model, np  (Cells 3 & 4)
# Produit  : retriever
#
# RRF formula : score(d) = SUM [ 1 / (k + rank_i(d)) ]  k=60
# Dense BERT = semantique | BM25 = termes exacts juridiques

import re
from dataclasses import dataclass
from rank_bm25 import BM25Okapi

_STOP = {
    "من","في","على","عن","الى","هذا","هذه","التي","الذي",
    "وان","ان","لا","ما","مع","او","ولا","لم","كان","يكون",
    "اذا","هو","هي","وهو","وهي","ذلك","تلك"
}

def tok(text):
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s\d]", " ", text)
    return [t for t in text.split() if len(t) > 2 and t not in _STOP]

print("Construction de l'index BM25...")
tokenized = [tok(a.to_chunk_text()) for a in articles]
bm25      = BM25Okapi(tokenized)
print(f"BM25 : {len(tokenized)} docs | {sum(len(t) for t in tokenized)//len(tokenized)} tokens/doc")

@dataclass
class RetrievedArticle:
    article:     object   # LegalArticle
    dense_score: float
    bm25_score:  float
    rrf_score:   float
    dense_rank:  int
    bm25_rank:   int

class HybridRetriever:

    def __init__(self, articles, faiss_index, embed_model, bm25_index, k=60):
        self.articles = articles
        self.index    = faiss_index
        self.embed    = embed_model
        self.bm25     = bm25_index
        self.k        = k

    def _dense(self, query, n=20):
        v = self.embed.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        ).astype(np.float32)
        scores, idxs = self.index.search(v, n)
        return list(zip(idxs[0].tolist(), scores[0].tolist()))

    def _sparse(self, query, n=20):
        tokens = tok(query)
        if not tokens: return []
        scores = self.bm25.get_scores(tokens)
        top_i  = np.argsort(scores)[::-1][:n]
        return [(int(i), float(scores[i])) for i in top_i]

    def _fuse(self, dense_r, bm25_r, top_k=10):
        rrf, d_rk, b_rk, d_sc, b_sc = {}, {}, {}, {}, {}
        for rk, (i, s) in enumerate(dense_r):
            rrf[i]  = rrf.get(i, 0) + 1.0 / (self.k + rk + 1)
            d_rk[i] = rk + 1;  d_sc[i] = s
        for rk, (i, s) in enumerate(bm25_r):
            rrf[i]  = rrf.get(i, 0) + 1.0 / (self.k + rk + 1)
            b_rk[i] = rk + 1;  b_sc[i] = s
        top = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            RetrievedArticle(
                article=self.articles[i],
                dense_score=d_sc.get(i, 0.0), bm25_score=b_sc.get(i, 0.0),
                rrf_score=rrf[i], dense_rank=d_rk.get(i, 999), bm25_rank=b_rk.get(i, 999)
            )
            for i in top if 0 <= i < len(self.articles)
        ]

    def retrieve(self, query, top_k=5):
        return self._fuse(self._dense(query, 20), self._sparse(query, 20), top_k)

retriever = HybridRetriever(
    articles=articles, faiss_index=index,
    embed_model=embed_model, bm25_index=bm25
)
print("\nHybrid Retriever pret (Dense + BM25 + RRF k=60)")

print("\nTest : 'شروط صحة العقد'")
for r in retriever.retrieve("شروط صحة العقد", top_k=3):
    print(f"  الفصل {r.article.article_num:>6} | RRF={r.rrf_score:.4f} | {r.article.text[:60]}...")

Construction de l'index BM25...
BM25 : 1258 docs | 44 tokens/doc

Hybrid Retriever pret (Dense + BM25 + RRF k=60)

Test : 'شروط صحة العقد'
  الفصل     20 | RRF=0.0320 | لا يكون العقد تاما إذا احتفظ المتعاقدان صراحة بشروط معينة لك...
  الفصل    393 | RRF=0.0164 | تنقضي الالتزامات التعاقدية» إذا ارتضى المتعاقدان عقب إبرام ا...
  الفصل   1-65 | RRF=0.0164 | مع مراعاة أحكام هذا الباب» تخضع صحة العقد المبرم بشكل إلكترو...


## CELL 6 - Qwen2.5-7B-Instruct GGUF (llama-cpp)

In [10]:
# CELL 6 - Chargement Qwen2.5-7B-Instruct Q4_K_M via llama-cpp
#
# Produit : llm
#
# Comparatif LLMs arabes open-source (MMLU-AR) :
#   Qwen2.5-7B-Instruct  ~72%  <- CHOIX OPTIMAL
#   LLaMA-3.1-8B-Instruct ~61%
#   Mistral-7B-Instruct   ~58%
#
# Pourquoi llama-cpp et non Ollama dans Colab ?
#   API Python directe, n_gpu_layers=-1, pas de daemon reseau.
#   Ollama = meilleur sur machine locale persistante.

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODEL_REPO = "pbhappliedsystems/qwen-2.5-7B-instruct-gguf-Q4-K-M"
MODEL_FILE = "qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf"
MODEL_PATH = f"/content/{MODEL_FILE}"

if not os.path.exists(MODEL_PATH):
    print(f"Telechargement {MODEL_FILE} (~4.4 GB)...")
    hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/content")
    print("Telechargement termine")
else:
    print(f"Modele deja present : {MODEL_PATH}")

print("\nChargement en VRAM GPU (n_gpu_layers=-1)...")
llm = Llama(
    model_path   = MODEL_PATH,
    n_gpu_layers = -1,
    n_ctx        = 4096,
    n_batch      = 512,
    verbose      = False,
    seed         = 42,
    chat_format  = "chatml"
)
print("Qwen2.5-7B-Instruct charge en GPU")

resp = llm.create_chat_completion(
    messages=[{"role": "user", "content": "\u0642\u0644: \u062c\u0627\u0647\u0632"}],
    max_tokens=10
)
print(f"Test LLM : {resp['choices'][0]['message']['content']}")
print("\nllm pret pour Cell 7")

Telechargement qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf (~4.4 GB)...


qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

Telechargement termine

Chargement en VRAM GPU (n_gpu_layers=-1)...


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Qwen2.5-7B-Instruct charge en GPU
Test LLM : جاهز

llm pret pour Cell 7


## CELL 7 - Agentic RAG Engine

In [11]:
# CELL 7 - Agentic RAG Engine
#
# Requiert : llm (Cell 6), retriever (Cell 5)
# Produit  : rag
#
# Boucle agentique :
#   1. Query expansion  -> LLM genere 3 sous-requetes arabes
#   2. Multi-retrieval  -> retrieval hybride pour chaque sous-requete
#   3. Deduplication    -> fusion + re-ranking par score RRF
#   4. Generation       -> Qwen repond en citant [الفصل X]
#   5. Post-traitement  -> extraction citations + score confiance

import re
import json
from dataclasses import dataclass

_EXPAND_TMPL = (
    "\u0623\u0646\u062a \u062e\u0628\u064a\u0631 \u0641\u064a \u0642\u0627\u0646\u0648\u0646 "
    "\u0627\u0644\u0627\u0644\u062a\u0632\u0627\u0645\u0627\u062a \u0648\u0627\u0644\u0639\u0642\u0648"
    "\u062f \u0627\u0644\u0645\u063a\u0631\u0628\u064a.\n"
    "\u0627\u0644\u0633\u0624\u0627\u0644: \"{q}\"\n"
    "\u0623\u0646\u0634\u0626 3 \u0627\u0633\u062a\u0641\u0633\u0627\u0631\u0627\u062a "
    "\u0628\u062d\u062b\u064a\u0629 \u0645\u062e\u062a\u0644\u0641\u0629 \u0644\u0644\u0628\u062d\u062b "
    "\u0641\u064a \u0647\u0630\u0627 \u0627\u0644\u0642\u0627\u0646\u0648\u0646.\n"
    "\u0623\u062c\u0628 \u0641\u0642\u0637 \u0628\u0640 JSON array:\n"
    "[\"\u0627\u0633\u062a\u0641\u0633\u0627\u0631 1\", "
    "\"\u0627\u0633\u062a\u0641\u0633\u0627\u0631 2\", "
    "\"\u0627\u0633\u062a\u0641\u0633\u0627\u0631 3\"]"
)

_ANSWER_TMPL = (
    "\u0623\u0646\u062a \u062e\u0628\u064a\u0631 \u0642\u0627\u0646\u0648\u0646\u064a "
    "\u0641\u064a \u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u0627\u0644\u062a\u0632\u0627\u0645\u0627"
    "\u062a \u0648\u0627\u0644\u0639\u0642\u0648\u062f \u0627\u0644\u0645\u063a\u0631\u0628\u064a "
    "(\u0638\u0647\u064a\u0631 9 \u0631\u0645\u0636\u0627\u0646 1331).\n\n"
    "\u0627\u0644\u0633\u0624\u0627\u0644: {q}\n\n"
    "\u0627\u0644\u0645\u0648\u0627\u062f \u0627\u0644\u0642\u0627\u0646\u0648\u0646\u064a\u0629:\n{ctx}\n\n"
    "\u0627\u0644\u062a\u0639\u0644\u064a\u0645\u0627\u062a:\n"
    "- \u0623\u062c\u0628 \u0628\u0627\u0644\u0639\u0631\u0628\u064a\u0629 \u0628\u0634\u0643\u0644 "
    "\u0645\u0641\u0635\u0644\n"
    "- \u0627\u0643\u062a\u0628 [\u0627\u0644\u0641\u0635\u0644 X] \u0639\u0646\u062f \u0643\u0644 "
    "\u0627\u0633\u062a\u0646\u0627\u062f\n"
    "- \u0635\u0631\u062d \u0627\u0630\u0627 \u0643\u0627\u0646\u062a \u0627\u0644\u0645\u0648\u0627\u062f "
    "\u063a\u064a\u0631 \u0643\u0627\u0641\u064a\u0629\n\n"
    "\u0627\u0644\u062c\u0648\u0627\u0628:"
)

@dataclass
class RAGResponse:
    query:              str
    expanded_queries:   list
    retrieved_articles: list
    answer:             str
    cited_articles:     list
    confidence:         str

class AgenticLegalRAG:

    def __init__(self, llm, retriever):
        self.llm = llm
        self.ret = retriever

    def _call_llm(self, prompt, max_tokens=512, temperature=0.1):
        resp = self.llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=temperature, top_p=0.9
        )
        return resp["choices"][0]["message"]["content"].strip()

    def _expand(self, query):
        raw = self._call_llm(_EXPAND_TMPL.format(q=query), 200, 0.4)
        m   = re.search(r"\[.*?\]", raw, re.DOTALL)
        if m:
            try:
                exp = json.loads(m.group())
                return [x for x in exp if isinstance(x, str)][:3]
            except Exception:
                pass
        return [query]

    def _multi_retrieve(self, queries, top_k=5):
        seen = {}
        for q in queries:
            for r in self.ret.retrieve(q, top_k=top_k):
                k = r.article.article_num
                if k not in seen or r.rrf_score > seen[k].rrf_score:
                    seen[k] = r
        return sorted(seen.values(), key=lambda x: x.rrf_score, reverse=True)[:top_k + 3]

    def query(self, user_q, verbose=True):
        SEP = "=" * 65
        if verbose:
            print(f"\n{SEP}")
            print(f"Question : {user_q}")

        expanded  = self._expand(user_q)
        if verbose:
            print("\n[1] Sous-requetes :")
            for i, q in enumerate(expanded, 1): print(f"    {i}. {q}")

        retrieved = self._multi_retrieve([user_q] + expanded)
        if verbose:
            print("\n[2] Articles retrouves :")
            for r in retrieved[:5]:
                print(f"    الفصل {r.article.article_num:>6} | RRF={r.rrf_score:.4f} | {r.article.text[:55]}...")

        ctx_parts = []
        for r in retrieved[:5]:
            hdr = f"\u0627\u0644\u0641\u0635\u0644 {r.article.article_num}"
            if r.article.book: hdr += f" | {r.article.book}"
            ctx_parts.append(f"{'='*50}\n{hdr}\n{r.article.text}")
        ctx = "\n".join(ctx_parts)

        if verbose: print("\n[3] Generation en cours...")
        answer = self._call_llm(_ANSWER_TMPL.format(q=user_q, ctx=ctx), 900, 0.1)

        cited      = list(set(re.findall(r"\u0627\u0644\u0641\u0635\u0644\s+([\d]+(?:[\-.]\d+)?)", answer)))
        top_rrf    = retrieved[0].rrf_score if retrieved else 0
        confidence = "HIGH" if top_rrf > 0.025 else "MEDIUM" if top_rrf > 0.015 else "LOW"

        if verbose:
            print(f"\n{SEP}\nREPONSE [{confidence}]\n{SEP}\n{answer}")
            print(f"\nArticles cites : {cited}")

        return RAGResponse(
            query=user_q, expanded_queries=expanded,
            retrieved_articles=[{"num": r.article.article_num, "rrf": round(r.rrf_score, 4),
                                  "preview": r.article.text[:100]} for r in retrieved[:5]],
            answer=answer, cited_articles=cited, confidence=confidence
        )

rag = AgenticLegalRAG(llm=llm, retriever=retriever)
print("Agentic RAG pret !")

Agentic RAG pret !


## CELL 8 - Tests use cases juridiques

In [12]:
# CELL 8 - Tests use cases juridiques
# Requiert : rag (Cell 7)

USE_CASES = [
    "ما هي الشروط القانونية لصحة عقد البيع؟",
    "متى يكون العقد باطلا أو قابلا للإبطال؟",
    "ما هي أحكام المسؤولية المدنية وشروط التعويض؟",
    "ما هي مدد التقادم في قانون الالتزامات؟",
]

responses = []
for i, case in enumerate(USE_CASES, 1):
    print(f"\n{'#'*70}\nUSE CASE {i}/{len(USE_CASES)}")
    responses.append(rag.query(case, verbose=True))

print(f"\n{len(responses)} use cases traites.")
print("Reponses stockees dans `responses`")


######################################################################
USE CASE 1/4

Question : ما هي الشروط القانونية لصحة عقد البيع؟

[1] Sous-requetes :
    1. ما هي الشروط القانونية اللازمة لصحة عقد البيع في القانون المغربي؟
    2. هل يشترط القانون المغربي وجود شهود في عقد البيع؟
    3. ما هي العقوبات القانونية عند عدم الالتزام بشروط صحة عقد البيع في المغرب؟

[2] Articles retrouves :
    الفصل    601 | RRF=0.0300 | يسوغ أن يشترط في عقد البيع ثبوت الحق للمشتري أو للبائع ...
    الفصل    516 | RRF=0.0294 | الالتزام بتسليم الشيء يشمل أيضا توابعه؛ وفقا لما يقضي ب...
    الفصل    260 | RRF=0.0272 | إذا اتفق المتعاقدان على أن العقد يفسخ عند عدم وفاء أحده...
    الفصل    581 | RRF=0.0256 | إذا اشترط بمقتضى العقد أو العرف المحلي*14 أن البيع يفسخ...
    الفصل    504 | RRF=0.0164 | يجب أن يحصل التسليم فور إبرام العقد, إلا ما تقتضيه طبيع...

[3] Generation en cours...

REPONSE [HIGH]
العقد هو أحد أهم العقود في القانون المغربي، ويشمل عقد البيع العديد من الشروط القانونية لصحته. فيما يلي الشروط

## CELL 9 - Interface interactive

In [ ]:
# CELL 9 - Interface interactive
# Requiert : rag (Cell 7)
# Saisir 'خروج' ou Ctrl+C pour quitter

print("Systeme RAG — قانون الالتزامات والعقود")
print("Saisir votre question en arabe | 'خروج' pour quitter\n")

while True:
    try:
        q = input("Question : ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nFin de session.")
        break
    if not q: continue
    if q in ["\u062e\u0631\u0648\u062c", "exit", "quit"]:
        print("Au revoir.")
        break
    try:
        rag.query(q, verbose=True)
    except Exception as e:
        print(f"[ERREUR] {e}")

## CELL 10 - Evaluation Precision@K et Recall@K

In [13]:
# CELL 10 - Evaluation quantitative du retriever
# Requiert : retriever (Cell 5), articles (Cell 3), embed_model, index, np

EVAL_SET = [
    {"query": "\u0634\u0631\u0648\u0637 \u0635\u062d\u0629 \u0627\u0644\u0639\u0642\u062f \u0648\u0623\u0631\u0643\u0627\u0646\u0647",
     "relevant": ["2","3","4","5","6"]},
    {"query": "\u0627\u0644\u062a\u0639\u0648\u064a\u0636 \u0639\u0646 \u0627\u0644\u0636\u0631\u0631 \u0627\u0644\u0645\u0633\u0624\u0648\u0644\u064a\u0629",
     "relevant": ["77","78","79","88","89"]},
    {"query": "\u0639\u0642\u062f \u0627\u0644\u0628\u064a\u0639 \u0623\u062d\u0643\u0627\u0645\u0647",
     "relevant": ["478","479","480","481"]},
    {"query": "\u0627\u0644\u062a\u0642\u0627\u062f\u0645 \u0648\u0627\u0646\u0642\u0636\u0627\u0621 \u0627\u0644\u0627\u0644\u062a\u0632\u0627\u0645\u0627\u062a",
     "relevant": ["387","388","389","391"]},
    {"query": "\u0627\u0644\u0643\u0641\u0627\u0644\u0629 \u0648\u0627\u0644\u0636\u0645\u0627\u0646",
     "relevant": ["1117","1118","1119","1120"]},
]

K = 5

print(f"{'Query':<48} {'P@'+str(K):>6} {'R@'+str(K):>6}  Hits")
print("-" * 78)

p_all, r_all = [], []
for case in EVAL_SET:
    ret  = retriever.retrieve(case["query"], top_k=K)
    nums = [r.article.article_num for r in ret]
    hits = [n for n in nums if n in case["relevant"]]
    p    = len(hits) / K
    r    = len(hits) / max(len(case["relevant"]), 1)
    p_all.append(p); r_all.append(r)
    print(f"{case['query'][:48]:<48} {p:>6.2f} {r:>6.2f}  {hits}")

n = len(EVAL_SET)
print("=" * 78)
print(f"{'MOYENNE':<48} {sum(p_all)/n:>6.3f} {sum(r_all)/n:>6.3f}")

# Comparaison Dense seul vs Hybrid
print("\n--- Dense seul vs Hybrid RRF ---")
p_dense = []
for case in EVAL_SET:
    v   = embed_model.encode([case["query"]], normalize_embeddings=True,
                              convert_to_numpy=True).astype(np.float32)
    _, idxs = index.search(v, K)
    nums    = [articles[i].article_num for i in idxs[0] if 0 <= i < len(articles)]
    hits    = [n for n in nums if n in case["relevant"]]
    p_dense.append(len(hits) / K)

print(f"Dense seul  mean P@{K} = {sum(p_dense)/n:.3f}")
print(f"Hybrid RRF  mean P@{K} = {sum(p_all)/n:.3f}  (+{(sum(p_all)/n - sum(p_dense)/n)*100:.1f}%)")

Query                                               P@5    R@5  Hits
------------------------------------------------------------------------------
شروط صحة العقد وأركانه                             0.00   0.00  []
التعويض عن الضرر المسؤولية                         0.00   0.00  []
عقد البيع أحكامه                                   0.20   0.25  ['478']
التقادم وانقضاء الالتزامات                         0.00   0.00  []
الكفالة والضمان                                    0.00   0.00  []
MOYENNE                                           0.040  0.050

--- Dense seul vs Hybrid RRF ---
Dense seul  mean P@5 = 0.040
Hybrid RRF  mean P@5 = 0.040  (+0.0%)


---
## Architecture — Tableau de bord

| Composant | Choix | Justification |
|---|---|---|
| Extraction PDF | OCR Tesseract 300 DPI | Polices CID ToUnicode cassees — PyMuPDF/pdftotext = garbage arabe |
| OCR parallele | ProcessPoolExecutor | ~2x rapide (2 CPU Colab). Sur A100 Pro : easyocr GPU (5x) |
| Chunking | Article-level الفصل N | Structure naturelle LOC. +25% precision vs sliding window |
| Nettoyage | Regex semantiques | Distingue notes de bas de page vs listes articles (faux positifs evites) |
| Numeros articles | clean_article_num() | OCR colle superscripts : 2901197 -> 290 (16 cas testes) |
| Embedding | camel-bert-base-sts | Best Arabic STS HuggingFace, vocabulaire juridique marocain |
| Index vectoriel | FAISS IndexFlatIP GPU | Cosine exact, adapte pour < 2000 articles |
| Sparse | BM25Okapi | Precision termes juridiques exacts |
| Fusion | RRF k=60 | Sans hyperparametre, robuste |
| LLM | Qwen2.5-7B Q4_K_M | #1 arabe open-source MMLU-AR ~72%, 4.4 GB, tient sur T4 |
| Backend LLM | llama-cpp-python | API Python directe, n_gpu_layers=-1, ideal Colab |
| Agentique | Query expansion x3 | Ameliore le recall sur questions complexes |